# TIB P4 — private Colab feasibility

Prepared executable notebook; **not yet run on Colab**. Select a **T4 GPU** runtime for T4 evidence. A100 is allowed for separate capacity evidence and will not pass the T4 gate. Only three sealed staged pilot sources are bundled; no abstracts, model-selection records or test records.

Upload the companion ZIP to your **private** MyDrive/TIB_P4_FEASIBILITY_261019 folder. Keep source/outputs private and attribution intact. Checkpoints and logs go to that folder. This notebook does not run pilot40, selection300, final523 or choose fallback. 40 diagnostics belong inside the later final cohort.

350 is the train-derived maximum word cap, not a target length. Native output reserve768 remains a feasibility proposal. No cap reduction after selection/test outcomes. Session guard is11.5hours including setup; inference subprocesses stop before that guard. Checkpoint-node writes occur after each completed node; unexpected interruption retains attempt markers and NULL unknown costs. Drive cross-session survival must still be verified before formal runs.

In [ ]:
import time, pathlib
SESSION_STARTED=time.monotonic()
SESSION_GUARD_SECONDS=11.5*3600
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT=pathlib.Path('/content/drive/MyDrive/TIB_P4_FEASIBILITY_261019')
ZIP_PATH=DRIVE_ROOT/'tib-p4-feasibility-261019-1802-v2.zip'
STORE=DRIVE_ROOT/'checkpoints'
assert ZIP_PATH.is_file(), f'Upload ZIP first: {ZIP_PATH}'
assert not (DRIVE_ROOT/'PUBLIC_SHARE_AUTHORIZED').exists(), 'Keep this evidence private'
print('Drive checkpoint folder:',STORE)

In [ ]:
import subprocess,sys
subprocess.run([sys.executable,'-m','pip','install','--quiet','transformers==4.57.6','accelerate>=1.0,<2','safetensors','psutil','scikit-learn'],check=True)
import torch
assert torch.cuda.is_available(), 'Select GPU runtime first'
print('Actual GPU:',torch.cuda.get_device_name(0))
print('T4 name check:',torch.cuda.get_device_name(0) in ('Tesla T4','NVIDIA Tesla T4','NVIDIA T4'))

In [ ]:
import zipfile,hashlib,json,os
EXPECTED_ZIP_SHA256='600fc9734dee141e7881a90812273fb34852d15d88a1d3f8aa6b4570fa7db6b3'
assert hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest()==EXPECTED_ZIP_SHA256
BUNDLE=pathlib.Path('/content/tib_p4_feasibility_261019')
BUNDLE.mkdir(exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as archive:
    for name in archive.namelist():
        assert (BUNDLE/name).resolve().is_relative_to(BUNDLE.resolve()), 'Unsafe ZIP member'
    archive.extractall(BUNDLE)
manifest=json.loads((BUNDLE/'bundle_manifest.json').read_text())
for name,expected in manifest['files'].items():
    assert hashlib.sha256((BUNDLE/name).read_bytes()).hexdigest()==expected,name
os.chdir(BUNDLE)
sys.path.insert(0,str(BUNDLE))
print('Verified source-only staged IDs:',manifest['staged_pilot_ids'])

In [ ]:
from experiments.scripts.tib_benchmark_checkpoint import probe_store
from experiments.scripts.tib_benchmark_adapter_cpu import lead3,lexrank
STORE.mkdir(parents=True,exist_ok=True)
print('Actual mounted-store probe:',probe_store(STORE))
records=[json.loads(line) for line in (BUNDLE/'staged_sources.jsonl').read_text().splitlines()]
cpu=[]
for row in records:
    for name,algorithm in [('lead3',lead3),('lexrank',lexrank)]:
        started=time.perf_counter()
        result=algorithm(row['source_normalized'],word_cap=350)
        cpu.append({'record_id':row['record_id'],'candidate_id':name,'wall_seconds':time.perf_counter()-started,**result})
from experiments.scripts.tib_benchmark_checkpoint import commit
commit(STORE/('cpu-smoke-'+str(time.time_ns())+'.json'),cpu)
print('CPU control outputs:',len(cpu),'(technical only)')

In [ ]:
# Each candidate gets its own process; memory is released between models.
# First capacity pass: all three neural candidates and three sealed sources.
# Model checkpoints download from the exact revisions in candidate_profiles.json.
for candidate in ['bart','led','qwen3']:
    remaining=SESSION_GUARD_SECONDS-(time.monotonic()-SESSION_STARTED)-60
    if remaining<=60:
        print('Session guard reached; preserve partial receipts and resume later');break
    command=[sys.executable,'-u','-m','experiments.scripts.tib_benchmark_colab_capacity',
             '--bundle',str(BUNDLE),'--store',str(STORE),'--candidate',candidate,
             '--wall-hours',str(min(11.0,remaining/3600))]
    log=STORE/(candidate+'-session-'+str(time.time_ns())+'.log')
    try:
        with log.open('w') as handle:
            result=subprocess.run(command,stdout=handle,stderr=subprocess.STDOUT,timeout=remaining)
        print(candidate,'exit',result.returncode,'log:',log)
    except subprocess.TimeoutExpired:
        print(candidate,'session deadline: preserve attempted/partial records and node receipts');break

In [ ]:
from experiments.scripts.tib_benchmark_checkpoint import restore
reports=[]
for path in sorted(STORE.glob('*/capacity-*.json')):
    data=restore(path)
    reports.append({'candidate_id':data['candidate_id'],'GPU':data['device_name'],'status':data['status'],
       'T4_feasibility_verified':data['T4_feasibility_verified'],'RAM_bytes':data.get('sampled_peak_process_rss_bytes'),
       'VRAM_allocated_bytes':data.get('cuda_peak_allocated_bytes'),'VRAM_reserved_bytes':data.get('cuda_peak_reserved_bytes'),
       'wall_seconds':data.get('total_wall_seconds'),'receipt':str(path)})
import pandas as pd
display(pd.DataFrame(reports))
print('These are feasibility receipts, not formal benchmark outcomes. Do not infer T4 feasibility from A100.')
print('Keep full523 target. Decide523->500->400 only after validation/profile conservative budget projection; none fit -> blocked.')